In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, importlib
ROOT = "/content/drive/MyDrive/simuteh-2330511068"
SRC = f"{ROOT}/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
os.chdir(f"{ROOT}/notebook")

import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from utils_cv import baca, tampil, kontras_bercak
print("OK", os.getcwd())

Mounted at /content/drive
OK /content/drive/MyDrive/simuteh-2330511068/notebook


In [2]:
%%writefile /content/drive/MyDrive/simuteh-2330511068/src/praproses.py
"""praproses.py: rangkaian praproses SIMUTEH (dipakai ulang mulai P5)."""
import json
import cv2
import numpy as np

PARAM_BAWAAN = {
    "lebar": 640,         # sampling: lebar citra kerja (piksel)
    "gamma": 1.0,         # < 1 mencerahkan, > 1 menggelapkan
    "clahe_clip": 2.0,    # 0 berarti CLAHE tidak dipakai
    "clahe_tile": 8,
    "penapis": "median",  # "none", "rerata", "gaussian", "median"
    "kernel": 3,
    "sigma": 1.0,
    "tajam_k": 0.0,       # 0 berarti tanpa penajaman
}


def praproses(img_bgr, p=None):
    """Urutan: ubah ukuran -> gamma -> CLAHE kanal V -> redam derau -> tajamkan."""
    p = {**PARAM_BAWAAN, **(p or {})}

    # 1. Ubah ukuran: lebar = p["lebar"], rasio aspek tetap, INTER_AREA
    h, w = img_bgr.shape[:2]
    lebar = int(p["lebar"])
    tinggi = int(round(h * lebar / w))
    img = cv2.resize(img_bgr, (lebar, tinggi), interpolation=cv2.INTER_AREA)

    # 2. Koreksi gamma lewat tabel pencarian (LUT)
    if p["gamma"] != 1:
        lut = np.array([255 * (i / 255) ** p["gamma"] for i in range(256)])
        lut = lut.clip(0, 255).astype(np.uint8)
        img = cv2.LUT(img, lut)

    # 3. CLAHE pada kanal V saja, supaya warna (H dan S) tidak berubah
    if p["clahe_clip"] > 0:
        ubin = int(p["clahe_tile"])
        clahe = cv2.createCLAHE(clipLimit=p["clahe_clip"],
                                tileGridSize=(ubin, ubin))
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        hsv[:, :, 2] = clahe.apply(hsv[:, :, 2])
        img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # 4. Redam derau
    k = int(p["kernel"])
    if k % 2 == 0:          # ukuran kernel harus ganjil
        k += 1
    if p["penapis"] == "rerata":
        img = cv2.blur(img, (k, k))
    elif p["penapis"] == "gaussian":
        img = cv2.GaussianBlur(img, (k, k), p["sigma"])
    elif p["penapis"] == "median":
        img = cv2.medianBlur(img, k)
    # "none": tidak ada penapisan

    # 5. Penajaman (unsharp masking), opsional
    if p["tajam_k"] > 0:
        blur = cv2.GaussianBlur(img, (5, 5), 1.0)
        img = cv2.addWeighted(img, 1 + p["tajam_k"], blur, -p["tajam_k"], 0)

    return img


def simpan_param(p, path="../hasil/param_praproses.json"):
    with open(path, "w") as f:
        json.dump({**PARAM_BAWAAN, **p}, f, indent=2)

Writing /content/drive/MyDrive/simuteh-2330511068/src/praproses.py


In [4]:
from praproses import praproses, simpan_param
importlib.reload(sys.modules["praproses"])
from praproses import praproses, simpan_param

# ISI sendiri berdasarkan Tabel 8, 10, 11 dan eksperimen CLAHE
param = {"gamma": 1.0, "clahe_clip": 2.0, "penapis": "median", "kernel": 3}
simpan_param(param, "../hasil/param_praproses.json")
KANAL = 2   # 0 = B, 1 = G, 2 = R. Sesuaikan dengan kanal terbaik Anda di Bagian 2

label = pd.read_csv("../data/sintetis/label.csv")
baris = []
for _, d in label[label.kelas != "sehat"].iterrows():
    img = baca(f"../data/sintetis/citra/{d.nama_file}")
    md = cv2.imread(f"../data/sintetis/mask_daun/{d.nama_file}", 0)
    mb = cv2.imread(f"../data/sintetis/mask_bercak/{d.nama_file}", 0)
    out = praproses(img, param)
    baris.append([d.kondisi_cahaya,
                  kontras_bercak(img[:, :, KANAL], md, mb),
                  kontras_bercak(out[:, :, KANAL], md, mb)])

evaluasi = pd.DataFrame(baris, columns=["kondisi", "KB_sebelum", "KB_sesudah"])
urutan = ["normal", "gelap", "terang", "kontras_rendah", "bayangan"]
t12 = evaluasi.groupby("kondisi").mean().round(2).reindex(urutan)
t12["perubahan_%"] = ((t12.KB_sesudah - t12.KB_sebelum) / t12.KB_sebelum * 100).round(1)
print(t12)
t12.to_csv("../hasil/b5_tabel12.csv")

                KB_sebelum  KB_sesudah  perubahan_%
kondisi                                            
normal                7.02        8.66         23.4
gelap                 9.24       15.57         68.5
terang                6.18        4.84        -21.7
kontras_rendah        3.68        4.70         27.7
bayangan              5.54        9.97         80.0


In [5]:
label = pd.read_csv("../data/sintetis/label.csv")
sehat = label[label.kelas == "sehat"]
print(len(sehat), "citra sehat")

baris = []
for _, d in sehat.iterrows():
    img = baca(f"../data/sintetis/citra/{d.nama_file}")
    md = cv2.imread(f"../data/sintetis/mask_daun/{d.nama_file}", 0)
    out = praproses(img, param)

    # mask harus sama ukurannya dengan citra hasil praproses
    if out.shape[:2] != md.shape:
        md = cv2.resize(md, (out.shape[1], out.shape[0]), interpolation=cv2.INTER_NEAREST)
    daun = md > 0

    sebelum = img[:, :, KANAL].astype(float)[daun] if img.shape[:2] == daun.shape else None
    if sebelum is None:
        img_k = cv2.resize(img, (out.shape[1], out.shape[0]), interpolation=cv2.INTER_AREA)
        sebelum = img_k[:, :, KANAL].astype(float)[daun]
    sesudah = out[:, :, KANAL].astype(float)[daun]

    # titik terang: piksel daun yang melebihi rerata + 3 simpangan baku daun SEBELUM praproses
    batas = sebelum.mean() + 3 * sebelum.std()
    baris.append([d.nama_file, d.kondisi_cahaya,
                  sebelum.std(), sesudah.std(),
                  100 * (sebelum > batas).mean(), 100 * (sesudah > batas).mean()])

uji = pd.DataFrame(baris, columns=["nama", "kondisi", "std_sebelum", "std_sesudah",
                                   "%terang_sebelum", "%terang_sesudah"])
urutan = ["normal", "gelap", "terang", "kontras_rendah", "bayangan"]
ringkas = uji.groupby("kondisi")[["std_sebelum", "std_sesudah",
                                  "%terang_sebelum", "%terang_sesudah"]].mean().round(2).reindex(urutan)
ringkas["perubahan_std_%"] = ((ringkas.std_sesudah - ringkas.std_sebelum) / ringkas.std_sebelum * 100).round(1)
print(ringkas)
uji.to_csv("../hasil/b5_uji_sehat_per_citra.csv", index=False)
ringkas.to_csv("../hasil/b5_uji_sehat.csv")

20 citra sehat
                std_sebelum  std_sesudah  %terang_sebelum  %terang_sesudah  \
kondisi                                                                      
normal                10.40         9.58             1.13             1.08   
gelap                  8.54         5.71             0.74             0.69   
terang                10.03        12.26             0.87             0.03   
kontras_rendah        10.06        10.77             0.19             0.05   
bayangan              15.58        11.36             0.42             0.23   

                perubahan_std_%  
kondisi                          
normal                     -7.9  
gelap                     -33.1  
terang                     22.2  
kontras_rendah              7.1  
bayangan                  -27.1  
